<a href="https://colab.research.google.com/github/LsYoli/fisica_computacional/blob/main/Taller_8/TALLER_8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 6. Taller 8: la radiación de cuerpo negro


### A. Enunciado y requisitos

La radiancia espectral de un cuerpo negro a temperatura $T$ está dada por la **ley de Planck**:

$$B_\lambda(\lambda, T) = \frac{2hc^2}{\lambda^5}\,\frac{1}{e^{hc/(\lambda k_B T)} - 1} ,$$

donde $B_\lambda$ es la potencia emitida por unidad de área, de ángulo sólido y de longitud de onda [W m⁻² sr⁻¹ m⁻¹], $\lambda$ la longitud de onda [m], $T$ la temperatura [K], $h = 6.62607015\times10^{-34}$ J s la constante de Planck, $c = 299792458$ m/s la rapidez de la luz en el vacío y $k_B = 1.380649\times10^{-23}$ J/K la constante de Boltzmann. El factor $2hc^2/\lambda^5$ cuenta los modos electromagnéticos con su energía y el factor con el exponencial es la ocupación de esos modos. Estas tres constantes son exactas en el Sistema Internacional.

**Variable adimensional.** Con $x = hc/(\lambda k_B T)$, la razón entre la energía de un fotón y la energía térmica, el espectro queda

$$B_\lambda \propto \frac{x^5}{e^x - 1} .$$

**Ley de desplazamiento de Wien.** El máximo del espectro cumple $dB_\lambda/dx = 0$, que se reduce a

$$x = n\left(1 - e^{-x}\right), \qquad n = 5 ,$$

ecuación trascendente sin solución en funciones elementales, con la raíz trivial $x = 0$ y una raíz positiva $x_5$. Esa raíz fija la constante de Wien $b$:

$$\lambda_{\max}\,T = b = \frac{hc}{k_B\,x_5} .$$

Si el espectro se expresa por unidad de frecuencia, $B_\nu \propto x^3/(e^x - 1)$, y el máximo cumple la misma ecuación con $n = 3$.

**Ley de Stefan-Boltzmann.** La potencia total emitida por unidad de área es $M = \int_0^\infty \pi B_\lambda\,d\lambda = \sigma T^4$, con

$$\sigma = \frac{2\pi k_B^4}{h^3 c^2}\,I, \qquad I = \int_0^\infty \frac{x^3}{e^x - 1}\,dx ,$$

donde $\sigma$ es la constante de Stefan-Boltzmann [W m⁻² K⁻⁴] e $I$ una integral pura, cuyo valor exacto es $\pi^4/15$. Es una integral impropia. El cambio de variable $x = t/(1 - t)$, con $dx = dt/(1-t)^2$, la lleva al intervalo finito $[0, 1]$:

$$I = \int_0^1 \frac{x(t)^3}{e^{x(t)} - 1}\,\frac{dt}{(1 - t)^2} .$$

**El taller tiene tres partes:**

| Parte | Lenguaje | Qué se hace |
|---|---|---|
| B | Fortran | Módulo con tres métodos para hallar ceros; constante de Wien |
| C | Fortran | Módulo de Romberg del Taller 7 con un integrando nuevo; constante de Stefan-Boltzmann |
| D | Python | Máximo del espectro solar en una malla, con spline, comparado con la ley de Wien |

**Requisitos generales**

1. Todo el trabajo, incluido este notebook, se guarda en una carpeta llamada `taller_cuerpo_negro`, con los nombres de archivo que se indican en cada parte. El archivo `modulo_romberg.f90` del Taller 7 se copia en esa carpeta.
2. Se usa doble precisión: `real(dp)` en Fortran y `float` (64 bits) en Python.
3. Los métodos para hallar ceros tienen un máximo de 100 iteraciones y terminan con un mensaje si no convergen. Romberg conserva el límite de niveles del Taller 7.
4. La bisección comprueba el cambio de signo antes de iterar. Newton y secante se detienen con un mensaje si el denominador es nulo.
5. Se compila con `-fcheck=bounds`.
6. Los números reales de los archivos `.dat` se escriben con formato `es24.16`, y los identificadores y conteos como enteros.

**Qué se entrega**

- En la carpeta del repositorio: los programas, los archivos de datos, la figura y este notebook, con la celda de verificación en 5 de 5 PASS y las respuestas de la sección E en celdas Markdown al final.
- En Moodle: el reporte de una página y el enlace a la carpeta.

<div style="border-left: 5px solid #B02A20; background-color: #FBEDEC; padding: 10px 14px; margin: 8px 0;">

**No está permitido** usar `scipy.optimize`, `scipy.integrate`, `np.gradient`, `np.trapz` ni funciones de biblioteca que hallen ceros, integren o deriven. `CubicSpline`, con sus métodos `.roots()` y `.derivative()`, se permite solo en la Parte D; las diferencias centrales se programan. Los métodos se programan a partir de las secciones 2 a 5. Cada estudiante sustentará su programa.

</div>

### B. Parte Fortran: constante de Wien

**Actividades**

1. Escriba `modulo_ceros.f90` con la función $F(x) = x - n(1 - e^{-x})$, su derivada y los métodos de bisección, Newton y secante. La organización en procedimientos es decisión suya; $n$ se pasa como argumento y debe respetar las reglas de la Semana 6.
2. Escriba `programa_wien.f90`, que use ese módulo para hallar $x_5$ y $x_3$ con tolerancia $\varepsilon = 10^{-10}$, con estos datos iniciales fijos:

| Método | Datos iniciales | Parada |
|---|---|---|
| Bisección | $[a, b] = [1, 10]$ | $b - c \le \varepsilon$ (sección 2) |
| Newton | $x_0 = 10$ | $\lvert x_{k+1} - x_k\rvert < \varepsilon$ |
| Secante | $x_0 = 10$, $x_1 = 9$ | $\lvert x_{k+1} - x_k\rvert < \varepsilon$ |

3. Repita Newton para $n = 5$ con $x_0 = 0.5$, y explique el resultado en el reporte.
4. Calcule la constante de Wien $b$ con $x_5$ obtenido por Newton.
5. Registre cada iteración de los tres métodos con $n = 5$ para analizar la convergencia. Las iteraciones de cada método se numeran desde 1 y en orden: la primera es la que produce el primer valor nuevo ($c_1$ en bisección, $x_1$ en Newton, $x_2$ en la secante).

**Qué preparar**

| Archivo | Contenido |
|---|---|
| `modulo_ceros.f90` | Función, derivada y los tres métodos, con conteo de iteraciones |
| `programa_wien.f90` | Usa el módulo; escribe los tres archivos `.dat` de esta parte |
| `resultado_wien.dat` | 7 filas: $n$, método, raíz, iteraciones. Método: 1 bisección, 2 Newton, 3 secante, para $n = 5$ y $n = 3$; la fila 7 es $n = 5$, método 4 (Newton con $x_0 = 0.5$) |
| `iteraciones_wien.dat` | Una fila por iteración con $n = 5$: método (1 a 3), número de iteración, valor de $x$ |
| `constante_wien.dat` | Una línea: $x_5$, $x_3$, $b$ [m K] |

<div style="border-left: 5px solid #2E7D32; background-color: #EAF5EA; padding: 10px 14px; margin: 8px 0;">

**Consejo.** La bisección para $n = 5$ debe usar el número de iteraciones que predice la fórmula de la sección 2. Calcúlelo antes de ejecutar el programa: si no coincide, revise el criterio de parada.

</div>

In [1]:
%%writefile modulo_ceros.f90
! ==============================================================
! modulo_ceros.f90
!
! Ecuación de Wien en forma adimensional:  g(x) = x - n (1 - e^-x) = 0
! El módulo reúne la función, su derivada y tres buscadores de
! raíces. Todos comparten la misma firma de salida:
!    cero       raíz aproximada
!    pasos      iteraciones realizadas
!    convergio  .true. si se cumplió el criterio de parada
! Si unidad_reg es distinta de 0, cada iteración se anota en esa
! unidad con el formato  (método, iteración, valor).
! ==============================================================
module modulo_ceros
  implicit none
  integer, parameter :: dp = kind(1.0d0)
  integer, parameter :: ITER_MAX = 100                    ! tope común a los tres métodos
  integer, parameter :: M_BISECCION = 1, M_NEWTON = 2, M_SECANTE = 3

contains

  ! g(x): su cero positivo es la raíz de Wien; x = 0 también es raíz.
  real(dp) function funcion_wien(x, n_wien)
    real(dp), intent(in) :: x
    integer,  intent(in) :: n_wien      ! 5 para B_lambda, 3 para B_nu
    funcion_wien = x - real(n_wien, dp)*(1.0_dp - exp(-x))
  end function funcion_wien

  ! g'(x) = 1 - n e^-x. Newton la necesita; se escribe a mano para no
  ! derivar numéricamente.
  real(dp) function derivada_wien(x, n_wien)
    real(dp), intent(in) :: x
    integer,  intent(in) :: n_wien
    derivada_wien = 1.0_dp - real(n_wien, dp)*exp(-x)
  end function derivada_wien

  ! --------------------------------------------------------------
  ! Bisección sobre [izq0, der0]. Se detiene cuando la distancia del
  ! punto medio al extremo derecho es <= tolerancia: ese valor es la
  ! cota del error, así que el número de pasos se puede predecir
  ! con log2((der0 - izq0)/tolerancia).
  ! --------------------------------------------------------------
  subroutine metodo_biseccion(n_wien, izq0, der0, tolerancia, cero, pasos, convergio, unidad_reg)
    integer,  intent(in)  :: n_wien, unidad_reg
    real(dp), intent(in)  :: izq0, der0, tolerancia
    real(dp), intent(out) :: cero
    integer,  intent(out) :: pasos
    logical,  intent(out) :: convergio
    real(dp) :: izq, der, medio, g_izq

    izq = izq0
    der = der0
    cero = izq
    pasos = 0
    convergio = .false.

    ! Sin cambio de signo no hay garantía de raíz dentro del intervalo.
    g_izq = funcion_wien(izq, n_wien)
    if (g_izq*funcion_wien(der, n_wien) >= 0.0_dp) then
       print '(a)', 'Bisección: el intervalo no encierra un cambio de signo.'
       return
    end if

    do while (pasos < ITER_MAX)
       pasos = pasos + 1
       medio = 0.5_dp*(izq + der)
       if (unidad_reg /= 0) write(unidad_reg, '(i2,1x,i4,1x,es24.16)') M_BISECCION, pasos, medio
       cero = medio
       if (der - medio <= tolerancia) then
          convergio = .true.
          return
       end if
       ! Se conserva la mitad donde g cambia de signo.
       if (g_izq*funcion_wien(medio, n_wien) <= 0.0_dp) then
          der = medio
       else
          izq = medio
          g_izq = funcion_wien(izq, n_wien)
       end if
    end do
    print '(a,i0,a)', 'Bisección: sin convergencia tras ', ITER_MAX, ' iteraciones.'
  end subroutine metodo_biseccion

  ! --------------------------------------------------------------
  ! Newton-Raphson desde x_ini. El error de la iteración anterior se
  ! estima con el tamaño del paso, por eso se compara |dx| con la
  ! tolerancia.
  ! --------------------------------------------------------------
  subroutine metodo_newton(n_wien, x_ini, tolerancia, cero, pasos, convergio, unidad_reg)
    integer,  intent(in)  :: n_wien, unidad_reg
    real(dp), intent(in)  :: x_ini, tolerancia
    real(dp), intent(out) :: cero
    integer,  intent(out) :: pasos
    logical,  intent(out) :: convergio
    real(dp) :: x_actual, pendiente, dx

    x_actual = x_ini
    cero = x_actual
    pasos = 0
    convergio = .false.

    do while (pasos < ITER_MAX)
       pasos = pasos + 1
       pendiente = derivada_wien(x_actual, n_wien)
       if (pendiente == 0.0_dp) then          ! tangente horizontal: no corta el eje
          print '(a)', 'Newton: derivada nula, se detiene el método.'
          return
       end if
       dx = -funcion_wien(x_actual, n_wien)/pendiente
       x_actual = x_actual + dx
       cero = x_actual
       if (unidad_reg /= 0) write(unidad_reg, '(i2,1x,i4,1x,es24.16)') M_NEWTON, pasos, x_actual
       if (abs(dx) < tolerancia) then
          convergio = .true.
          return
       end if
    end do
    print '(a,i0,a)', 'Newton: sin convergencia tras ', ITER_MAX, ' iteraciones.'
  end subroutine metodo_newton

  ! --------------------------------------------------------------
  ! Secante con dos puntos de arranque. Reemplaza la derivada de
  ! Newton por la pendiente de la recta que une los dos últimos
  ! puntos; cuesta una evaluación de g por paso en lugar de dos.
  ! --------------------------------------------------------------
  subroutine metodo_secante(n_wien, x_prev0, x_act0, tolerancia, cero, pasos, convergio, unidad_reg)
    integer,  intent(in)  :: n_wien, unidad_reg
    real(dp), intent(in)  :: x_prev0, x_act0, tolerancia
    real(dp), intent(out) :: cero
    integer,  intent(out) :: pasos
    logical,  intent(out) :: convergio
    real(dp) :: x_prev, x_act, g_prev, g_act, dx

    x_prev = x_prev0
    x_act = x_act0
    cero = x_act
    pasos = 0
    convergio = .false.
    g_prev = funcion_wien(x_prev, n_wien)

    do while (pasos < ITER_MAX)
       pasos = pasos + 1
       g_act = funcion_wien(x_act, n_wien)
       if (g_act == g_prev) then              ! recta secante horizontal
          print '(a)', 'Secante: denominador nulo, se detiene el método.'
          return
       end if
       dx = -g_act*(x_act - x_prev)/(g_act - g_prev)
       x_prev = x_act
       g_prev = g_act
       x_act = x_act + dx
       cero = x_act
       if (unidad_reg /= 0) write(unidad_reg, '(i2,1x,i4,1x,es24.16)') M_SECANTE, pasos, x_act
       if (abs(dx) < tolerancia) then
          convergio = .true.
          return
       end if
    end do
    print '(a,i0,a)', 'Secante: sin convergencia tras ', ITER_MAX, ' iteraciones.'
  end subroutine metodo_secante

end module modulo_ceros

Writing modulo_ceros.f90


In [2]:
%%writefile programa_wien.f90
! ==============================================================
! programa_wien.f90
!
! Calcula las raíces positivas x5 (espectro por longitud de onda)
! y x3 (por frecuencia) con los tres métodos del módulo, y con x5
! obtiene la constante de Wien  b = h c / (k_B x5).
!
! Archivos que escribe:
!    resultado_wien.dat    n, método, raíz, iteraciones (7 filas)
!    iteraciones_wien.dat  historial de los tres métodos para n = 5
!    constante_wien.dat    x5, x3, b
! ==============================================================
program programa_wien
  use modulo_ceros
  implicit none

  real(dp), parameter :: TOL = 1.0e-10_dp
  real(dp), parameter :: H_PLANCK = 6.62607015e-34_dp      ! [J s]
  real(dp), parameter :: C_LUZ = 299792458.0_dp            ! [m/s]
  real(dp), parameter :: K_BOLTZMANN = 1.380649e-23_dp     ! [J/K]
  integer,  parameter :: ordenes(2) = [5, 3]

  real(dp) :: cero, x5, x3, b_wien
  integer  :: caso, n_wien, pasos
  integer  :: f_resultado, f_historial, f_constante, reg
  logical  :: convergio
  character(len=*), parameter :: FMT_RES = '(i3,1x,i3,1x,es24.16,1x,i4)'

  open(newunit=f_resultado, file='resultado_wien.dat',   status='replace', action='write')
  open(newunit=f_historial, file='iteraciones_wien.dat', status='replace', action='write')
  open(newunit=f_constante, file='constante_wien.dat',   status='replace', action='write')

  do caso = 1, size(ordenes)
     n_wien = ordenes(caso)

     ! El historial pedido es solo el de n = 5; con reg = 0 los métodos no escriben.
     reg = 0
     if (n_wien == 5) reg = f_historial

     call metodo_biseccion(n_wien, 1.0_dp, 10.0_dp, TOL, cero, pasos, convergio, reg)
     write(f_resultado, FMT_RES) n_wien, M_BISECCION, cero, pasos
     print '(a,i2,a,f18.12,i5)', 'n =', n_wien, '  bisección ', cero, pasos

     call metodo_newton(n_wien, 10.0_dp, TOL, cero, pasos, convergio, reg)
     write(f_resultado, FMT_RES) n_wien, M_NEWTON, cero, pasos
     print '(a,i2,a,f18.12,i5)', 'n =', n_wien, '  Newton    ', cero, pasos
     ! La raíz de Newton es la que se usa para x5 y x3.
     if (n_wien == 5) x5 = cero
     if (n_wien == 3) x3 = cero

     call metodo_secante(n_wien, 10.0_dp, 9.0_dp, TOL, cero, pasos, convergio, reg)
     write(f_resultado, FMT_RES) n_wien, M_SECANTE, cero, pasos
     print '(a,i2,a,f18.12,i5)', 'n =', n_wien, '  secante   ', cero, pasos
  end do

  ! Fila 7: Newton desde x0 = 0.5, que cae en la raíz trivial x = 0.
  call metodo_newton(5, 0.5_dp, TOL, cero, pasos, convergio, 0)
  write(f_resultado, FMT_RES) 5, 4, cero, pasos
  print '(a,es12.4,i5)', 'n = 5  Newton x0=0.5: ', cero, pasos

  b_wien = H_PLANCK*C_LUZ/(K_BOLTZMANN*x5)
  write(f_constante, '(3(es24.16,1x))') x5, x3, b_wien
  print '(a,es22.14)', 'b [m K] = ', b_wien

  close(f_resultado)
  close(f_historial)
  close(f_constante)
end program programa_wien

Writing programa_wien.f90


In [4]:
# Compila y ejecuta la Parte B (ejecute esta celda cuando sus archivos estén listos)
!gfortran -fcheck=bounds -o programa_wien modulo_ceros.f90 programa_wien.f90 && ./programa_wien

n = 5  bisección     4.965114231796   37
n = 5  Newton        4.965114231744    4
n = 5  secante       4.965114231744    5
n = 3  bisección     2.821439372106   37
n = 3  Newton        2.821439372122    5
n = 3  secante       2.821439372122    6
n = 5  Newton x0=0.5:   8.9452E-17    6
b [m K] =   2.89777195518517E-03


### C. Parte Fortran: constante de Stefan-Boltzmann

**Actividades**

1. Agregue a su `modulo_romberg.f90` del Taller 7 los integrandos de esta parte, sin cambiar la integración ni su criterio de parada. Los extremos $t = 0$ y $t = 1$ deben dar el límite correcto del integrando, que es cero en ambos.
2. Escriba `programa_sigma.f90`, que calcule $I$ en $t \in [0, 1]$ con tolerancia $10^{-8}$ y obtenga $\sigma$.
3. Calcule también $\int_0^{100} x^3/(e^x - 1)\,dx$ con el mismo Romberg y la misma tolerancia, **sin** cambio de variable.
4. Compare los dos resultados con $\pi^4/15$ y explique en el reporte lo que ocurre en el paso 3.

**Qué preparar**

| Archivo | Contenido |
|---|---|
| `modulo_romberg.f90` | Módulo del Taller 7 con los integrandos nuevos |
| `programa_sigma.f90` | Usa el módulo; escribe el archivo `.dat` de esta parte |
| `resultado_sigma.dat` | Dos filas, $k_{\text{final}}$, evaluaciones, integral, $\sigma$: fila 1 con cambio de variable, fila 2 sin cambio de variable. El nivel $k = 0$ es el trapecio inicial y las evaluaciones son $2^k + 1$ |

<div style="border-left: 5px solid #2E7D32; background-color: #EAF5EA; padding: 10px 14px; margin: 8px 0;">

**Consejo.** Para $t$ cercano a 1, $x$ es tan grande que $e^x$ desborda la doble precisión a partir de $x \approx 709$. El integrando es prácticamente cero mucho antes: decida a partir de qué valor de $x$ puede devolver cero.

</div>

In [3]:
%%writefile modulo_romberg.f90
! ============================================================
! Módulo:    modulo_romberg
! Propósito: integración de Romberg con conteo de evaluaciones
!            e integrandos del Taller 7.
! ============================================================
module modulo_romberg
  implicit none
  integer, parameter :: dp = kind(1.0d0)
  real(dp), parameter :: pi = 4*atan(1.0_dp)
  real(dp), parameter :: gamma = 0.5_dp          ! amortiguamiento [1/s]
  real(dp), parameter :: omega = 2*pi            ! frecuencia angular [rad/s]
  real(dp), parameter :: omega0_2 = omega**2 + gamma**2
  integer, parameter :: ID_ENERGIA = 1           ! integrando 2*gamma*v(t)^2
  integer, parameter :: ID_PI = 2                ! integrando 4/(1+x^2)
  integer, parameter :: ID_NEGRO_T = 3           ! cuerpo negro en t de [0,1], con x = t/(1-t)
  integer, parameter :: ID_NEGRO_X = 4           ! cuerpo negro directo en x
  real(dp), parameter :: X_MAX = 600.0_dp        ! para x > X_MAX el integrando vale menos que 1e-250
contains

  ! ----------------------------------------------------------
  ! posicion: x(t) = exp(-gamma t) cos(omega t)
  ! ----------------------------------------------------------
  real(dp) function posicion(t)
    real(dp), intent(in) :: t
    posicion = exp(-gamma*t)*cos(omega*t)
  end function posicion

  ! ----------------------------------------------------------
  ! velocidad: v(t) exacta del oscilador amortiguado
  ! ----------------------------------------------------------
  real(dp) function velocidad(t)
    real(dp), intent(in) :: t
    velocidad = -exp(-gamma*t)*(gamma*cos(omega*t) + omega*sin(omega*t))
  end function velocidad

  ! ----------------------------------------------------------
  ! energia: E(t) = v^2/2 + omega0^2 x^2/2   [J/kg]
  ! ----------------------------------------------------------
  real(dp) function energia(t)
    real(dp), intent(in) :: t
    energia = 0.5_dp*velocidad(t)**2 + 0.5_dp*omega0_2*posicion(t)**2
  end function energia

  ! ----------------------------------------------------------
  ! cuerpo_negro_x: x^3/(e^x - 1), escrito como x^3 e^(-x)/(1 - e^(-x)).
  ! Con esta forma e^(-x) solo puede subir hasta 1 (nunca desborda);
  ! en x = 0 el cociente es 0/0, pero el límite es 0 (vale ~x^2).
  ! Para x grande el valor real es ~0, por eso se corta en X_MAX.
  ! ----------------------------------------------------------
  real(dp) function cuerpo_negro_x(x)
    real(dp), intent(in) :: x
    real(dp) :: decae
    if (x <= 0.0_dp .or. x > X_MAX) then
      cuerpo_negro_x = 0.0_dp
    else
      decae = exp(-x)
      cuerpo_negro_x = x**3*decae/(1.0_dp - decae)
    end if
  end function cuerpo_negro_x

  ! ----------------------------------------------------------
  ! cuerpo_negro_t: el mismo integrando en la variable t de [0,1].
  ! x = t/(1-t)  ->  dx = dt/(1-t)^2. En t = 0 y t = 1 el límite es 0.
  ! ----------------------------------------------------------
  real(dp) function cuerpo_negro_t(t)
    real(dp), intent(in) :: t
    real(dp) :: u
    if (t <= 0.0_dp .or. t >= 1.0_dp) then
      cuerpo_negro_t = 0.0_dp
    else
      u = 1.0_dp - t
      cuerpo_negro_t = cuerpo_negro_x(t/u)/(u*u)
    end if
  end function cuerpo_negro_t

  ! ----------------------------------------------------------
  ! integrando: elige el integrando según id
  ! Entradas: id (ID_ENERGIA, ID_PI, ID_NEGRO_T o ID_NEGRO_X), x
  ! ----------------------------------------------------------
  real(dp) function integrando(id, x)
    integer, intent(in)  :: id
    real(dp), intent(in) :: x
    select case (id)
    case (ID_ENERGIA)
      integrando = 2*gamma*velocidad(x)**2
    case (ID_PI)
      integrando = 4.0_dp/(1.0_dp + x*x)
    case (ID_NEGRO_T)
      integrando = cuerpo_negro_t(x)
    case (ID_NEGRO_X)
      integrando = cuerpo_negro_x(x)
    case default
      error stop 'integrando: id desconocido'
    end select
  end function integrando

  ! ----------------------------------------------------------
  ! romberg: integración de Romberg de f(id, x) en [a, b]
  ! Entradas: id, a, b, tol, kmax
  ! Salidas:  S(0:kmax,0:kmax) tabla (solo el triángulo 0..k_final),
  !           k_final nivel alcanzado, n_eval evaluaciones de f
  ! ----------------------------------------------------------
  subroutine romberg(id, a, b, tol, kmax, S, k_final, n_eval)
    integer, intent(in)  :: id, kmax
    real(dp), intent(in) :: a, b, tol
    real(dp), allocatable, intent(out) :: S(:,:)
    integer, intent(out) :: k_final, n_eval
    real(dp) :: h, suma
    integer :: k, j, i

    allocate(S(0:kmax, 0:kmax))
    S = 0.0_dp
    h = b - a
    S(0,0) = 0.5_dp*h*(integrando(id, a) + integrando(id, b))
    n_eval = 2
    k_final = 0
    do k = 1, kmax
       h = h/2
       ! puntos nuevos del nivel k: a + (2i-1)h, i = 1..2^(k-1)
       suma = 0.0_dp
       do i = 1, 2**(k-1)
          suma = suma + integrando(id, a + (2*i - 1)*h)
       end do
       n_eval = n_eval + 2**(k-1)
       S(k,0) = 0.5_dp*S(k-1,0) + h*suma
       do j = 1, k
          S(k,j) = (4.0_dp**j*S(k,j-1) - S(k-1,j-1))/(4.0_dp**j - 1.0_dp)
       end do
       k_final = k
       if (abs(S(k,k) - S(k-1,k-1)) < tol) exit
    end do
  end subroutine romberg

end module modulo_romberg

Writing modulo_romberg.f90


In [5]:
%%writefile programa_sigma.f90
! ==============================================================
! programa_sigma.f90
!
! Constante de Stefan-Boltzmann  sigma = (2 pi k_B^4 / (h^3 c^2)) I,
! con  I = Int_0^inf x^3/(e^x - 1) dx = pi^4/15  (valor exacto).
! I se calcula con Romberg de dos maneras:
!    fila 1: cambio de variable x = t/(1-t), integral en [0, 1]
!    fila 2: directo en [0, 100], sin cambio de variable
! La fila 2 sirve para ver cómo el criterio de parada puede
! cumplirse antes de que la integral haya convergido.
!
! Salida: resultado_sigma.dat  (nivel final, evaluaciones, I, sigma)
! ==============================================================
program programa_sigma
  use modulo_romberg
  implicit none

  real(dp), parameter :: H_PLANCK = 6.62607015e-34_dp      ! [J s]
  real(dp), parameter :: C_LUZ = 299792458.0_dp            ! [m/s]
  real(dp), parameter :: K_BOLTZMANN = 1.380649e-23_dp     ! [J/K]
  real(dp), parameter :: TOL = 1.0e-8_dp
  integer,  parameter :: NIVEL_MAX = 20

  ! Prefactor de sigma; solo falta multiplicar por la integral.
  real(dp), parameter :: PREFACTOR = 2*pi*K_BOLTZMANN**4/(H_PLANCK**3*C_LUZ**2)
  real(dp), parameter :: I_EXACTA = pi**4/15

  ! Dos casos: (integrando, límite inferior, límite superior)
  integer,  parameter :: ids(2) = [ID_NEGRO_T, ID_NEGRO_X]
  real(dp), parameter :: lim_inf(2) = [0.0_dp, 0.0_dp]
  real(dp), parameter :: lim_sup(2) = [1.0_dp, 100.0_dp]

  real(dp), allocatable :: tabla(:,:)
  real(dp) :: valor_I, sigma_sb
  integer  :: fila, nivel_final, n_evaluaciones, f_salida

  open(newunit=f_salida, file='resultado_sigma.dat', status='replace', action='write')

  do fila = 1, 2
     call romberg(ids(fila), lim_inf(fila), lim_sup(fila), TOL, NIVEL_MAX, &
                  tabla, nivel_final, n_evaluaciones)

     ! La mejor estimación es el último elemento de la diagonal.
     valor_I = tabla(nivel_final, nivel_final)
     sigma_sb = PREFACTOR*valor_I

     write(f_salida, '(i3,1x,i8,1x,es24.16,1x,es24.16)') &
           nivel_final, n_evaluaciones, valor_I, sigma_sb
     print '(a,i1,a,i3,i8,es22.14,es22.14)', 'fila ', fila, ': nivel, evaluaciones, I, sigma =', &
           nivel_final, n_evaluaciones, valor_I, sigma_sb
     print '(a,es10.3)', '   |I - pi^4/15| = ', abs(valor_I - I_EXACTA)

     deallocate(tabla)      ! romberg reserva la tabla en cada llamada
  end do

  close(f_salida)
end program programa_sigma

Writing programa_sigma.f90


In [6]:
# Compila y ejecuta la Parte C (ejecute esta celda cuando sus archivos estén listos)
!gfortran -fcheck=bounds -o programa_sigma modulo_romberg.f90 programa_sigma.f90 && ./programa_sigma

fila 1: nivel, evaluaciones, I, sigma = 10    1025  6.49393940226710E+00  5.67037441918467E-08
   |I - pi^4/15| =  2.700E-13
fila 2: nivel, evaluaciones, I, sigma =  1       3  1.60729153996993E-15  1.40345393879648E-23
   |I - pi^4/15| =  6.494E+00


### D. Parte Python: máximo del espectro solar

Se modela el Sol como un cuerpo negro a $T = 5772$ K. El espectro se muestrea en una malla uniforme de longitudes de onda en $[100, 3000]$ nm, con paso $\Delta\lambda$.

**Actividades**

1. Escriba `maximo_espectro.py`. Construya $B_\lambda$ para $\Delta\lambda = 100, 50, 25$ y $12.5$ nm.
2. Para cada malla, obtenga $\lambda_{\max}$ de tres maneras: (a) mayor valor del arreglo; (b) cero del spline de la derivada central; (c) cero de la derivada del spline de $B_\lambda$. En (b) y (c), conserve los ceros donde la segunda derivada es negativa y, si queda más de uno, el más cercano al resultado de (a): en una malla gruesa el spline oscila y puede producir ceros espurios.
3. Lea $b$ de `constante_wien.dat` y obtenga el error absoluto en nm de cada resultado frente a $\lambda_{\max} = b/T$.
4. Escriba `maximo_espectro.dat` y grafique en `espectro_solar.png` el espectro de la malla de 25 nm, con $\lambda_{\max}$ marcado. Agregue en la misma figura el $\lambda$ que corresponde al máximo en frecuencia, $\lambda = hc/(k_B T x_3)$.

**Qué preparar**

| Archivo | Contenido |
|---|---|
| `maximo_espectro.py` | Construye los datos, halla los máximos, escribe los archivos y la figura |
| `maximo_espectro.dat` | Cuatro filas, una por $\Delta\lambda$: $\Delta\lambda$ [nm], error (a), error (b), error (c) [nm] |
| `espectro_solar.png` | Espectro con $\lambda_{\max}$ y el $\lambda$ del máximo en frecuencia |

In [7]:
%%writefile maximo_espectro.py
"""Parte D del Taller 8: máximo del espectro solar.

El Sol se modela como cuerpo negro a 5772 K. Se muestrea la radiancia
espectral en mallas uniformes de 100 a 3000 nm y, para cada paso, se
localiza el máximo con tres estrategias de costo creciente:

    A. el mayor valor del arreglo (limitado por el paso de la malla),
    B. el cero de un spline ajustado a la derivada central,
    C. el cero de la derivada de un spline ajustado al espectro.

Cada resultado se compara con lambda_max = b/T, con b leído del
archivo que escribió la Parte B.
Salidas: maximo_espectro.dat y espectro_solar.png.
"""
import numpy as np
import matplotlib
matplotlib.use("Agg")                      # sin ventana: solo se guarda la figura
import matplotlib.pyplot as plt
from scipy.interpolate import CubicSpline

# Constantes del SI (valores exactos)
H_PLANCK = 6.62607015e-34       # [J s]
C_LUZ = 299792458.0             # [m/s]
K_BOLTZMANN = 1.380649e-23      # [J/K]

T_SOL = 5772.0                  # [K]
LAMBDA_MIN, LAMBDA_MAX = 100.0, 3000.0          # [nm]
PASOS_NM = (100.0, 50.0, 25.0, 12.5)            # orden exigido en el .dat
PASO_FIGURA = 25.0


def radiancia_espectral(lam_nm):
    """Ley de Planck B_lambda(lambda, T_SOL) en W m^-2 sr^-1 m^-1.

    Se usa expm1 (e^y - 1) porque conserva precisión cuando el
    exponente es pequeño, es decir, para longitudes de onda largas.
    """
    lam = lam_nm*1e-9                                   # nm -> m
    exponente = H_PLANCK*C_LUZ/(lam*K_BOLTZMANN*T_SOL)
    return 2*H_PLANCK*C_LUZ**2/lam**5/np.expm1(exponente)


def diferencia_central(valores, paso):
    """Derivada central en los puntos interiores: (y[i+1] - y[i-1])/(2 paso).

    Devuelve dos elementos menos que la entrada, porque los extremos
    de la malla no tienen vecino a ambos lados.
    """
    return (valores[2:] - valores[:-2])/(2*paso)


def maximo_del_spline(spline_derivada, lam_arreglo):
    """Elige el máximo de entre los ceros de una derivada ya ajustada.

    Un máximo tiene segunda derivada negativa. Si una malla gruesa hace
    oscilar el spline y deja varios, se toma el más cercano al máximo
    del arreglo (lam_arreglo), que ya está cerca del correcto.
    """
    ceros = spline_derivada.roots(extrapolate=False)
    ceros = ceros[np.isfinite(ceros)]
    ceros = ceros[spline_derivada(ceros, 1) < 0]
    if ceros.size == 0:
        raise ValueError("el spline no tiene ningún máximo")
    return ceros[np.argmin(np.abs(ceros - lam_arreglo))]


def estimar_maximos(paso):
    """Construye la malla de un paso dado y devuelve (lam, B, A, B_, C)."""
    lam = np.arange(LAMBDA_MIN, LAMBDA_MAX + paso/2, paso)
    B = radiancia_espectral(lam)

    est_a = lam[np.argmax(B)]

    derivada = diferencia_central(B, paso)
    est_b = maximo_del_spline(CubicSpline(lam[1:-1], derivada), est_a)

    est_c = maximo_del_spline(CubicSpline(lam, B).derivative(), est_a)
    return lam, B, est_a, est_b, est_c


def graficar(lam, B, lam_wien, lam_frec, ruta):
    """Espectro de la malla elegida con los dos máximos marcados."""
    fig, ax = plt.subplots(figsize=(7.5, 4.2))
    ax.fill_between(lam, B*1e-12, color="#C9D3E8", alpha=0.6)
    ax.plot(lam, B*1e-12, color="#162A54", lw=1.6,
            label=f"$B_\\lambda$ (malla de {PASO_FIGURA:g} nm)")
    ax.axvline(lam_wien, color="#B02A20", lw=1.4,
               label=f"máximo en $\\lambda$: {lam_wien:.0f} nm")
    ax.axvline(lam_frec, color="#2E7D32", lw=1.4, ls="--",
               label=f"máximo en frecuencia: {lam_frec:.0f} nm")
    ax.set_xlim(LAMBDA_MIN, LAMBDA_MAX)
    ax.set_ylim(bottom=0)
    ax.set_xlabel("longitud de onda $\\lambda$ [nm]")
    ax.set_ylabel("$B_\\lambda$  [$10^{12}$ W m$^{-2}$ sr$^{-1}$ m$^{-1}$]")
    ax.set_title(f"Radiación de cuerpo negro del Sol, $T = {T_SOL:.0f}$ K")
    ax.legend(loc="upper right")
    fig.tight_layout()
    fig.savefig(ruta, dpi=150)
    plt.close(fig)


def main():
    # Parte B: x5, x3 y b [m K]
    _, x3, b_wien = np.loadtxt("constante_wien.dat")
    lam_ref = b_wien/T_SOL*1e9                         # lambda_max de referencia [nm]
    lam_frec = H_PLANCK*C_LUZ/(K_BOLTZMANN*T_SOL*x3)*1e9   # máximo de B_nu, en nm

    errores = np.zeros((len(PASOS_NM), 4))
    for i, paso in enumerate(PASOS_NM):
        _, _, est_a, est_b, est_c = estimar_maximos(paso)
        errores[i] = [paso, abs(est_a - lam_ref), abs(est_b - lam_ref), abs(est_c - lam_ref)]

    np.savetxt("maximo_espectro.dat", errores, fmt="%24.16E")

    print(f"lambda_max de referencia (b/T) = {lam_ref:.4f} nm")
    print(f"lambda del máximo en frecuencia = {lam_frec:.2f} nm")
    print("  paso [nm]   error A     error B     error C   [nm]")
    for paso, ea, eb, ec in errores:
        print(f"{paso:9.1f}{ea:12.3e}{eb:12.3e}{ec:12.3e}")
    razones = errores[:-1, 2:]/errores[1:, 2:]
    print("razón de errores al reducir el paso a la mitad (columnas B y C):")
    print(np.round(razones, 2))

    lam, B, *_ = estimar_maximos(PASO_FIGURA)
    graficar(lam, B, lam_ref, lam_frec, "espectro_solar.png")


if __name__ == "__main__":
    main()

Writing maximo_espectro.py


In [8]:
# Ejecuta la Parte D (ejecute esta celda cuando su archivo esté listo)
!python maximo_espectro.py

lambda_max de referencia (b/T) = 502.0395 nm
lambda del máximo en frecuencia = 883.48 nm
  paso [nm]   error A     error B     error C   [nm]
    100.0   2.039e+00   1.290e+01   1.202e+00
     50.0   2.039e+00   3.393e+00   3.216e-02
     25.0   2.039e+00   8.517e-01   2.905e-03
     12.5   2.039e+00   2.131e-01   3.127e-04
razón de errores al reducir el paso a la mitad (columnas B y C):
[[ 3.8  37.38]
 [ 3.98 11.07]
 [ 4.    9.29]]


### E. Preguntas de interpretación y reporte

Responda en celdas Markdown al final de este notebook, con los números de sus archivos:

1. **Costo de los métodos.** Compare las iteraciones de los tres métodos con $n = 5$. Con `iteraciones_wien.dat`, estime el orden de Newton y de la secante.
2. **Valor inicial.** ¿A qué valor llega Newton desde $x_0 = 0.5$? ¿Por qué es una raíz correcta de la ecuación, pero no la respuesta al problema físico?
3. **Parada prematura.** En la Parte C, ¿qué valor da Romberg en $[0, 100]$ y en qué nivel se detiene? Explique, con los puntos que evalúa en sus primeros niveles, por qué el criterio de la diagonal se cumple sin que la integral haya convergido. Proponga una comprobación adicional.
4. **Eslabón más débil.** En la Parte D, ¿con qué razón bajan los errores de (b) y de (c) al reducir $\Delta\lambda$ a la mitad? ¿Qué limita la precisión de (b)? ¿Por qué el error de (a) no baja en estas mallas?
5. **Dos máximos.** ¿Por qué el máximo en longitud de onda y el máximo en frecuencia no corresponden al mismo color? ¿Cuál de los dos es "el" color del Sol?

**Reporte (una página, plantilla del curso).** Las secciones de contexto físico, método y resultado se refieren a la constante de Wien y al máximo del espectro (Partes B y D). La Parte C puede alimentar la sección de obstáculo o decisión de implementación. El reporte es una síntesis y no repite las cinco respuestas.

**RESPUESTAS**

**1. Costo de los métodos.** Con $n = 5$, la bisección necesitó 37 iteraciones, la secante 5 y Newton 4. Las 37 de la bisección coinciden con la fórmula $\log_2(9/10^{-10}) \approx 36.4$, porque en cada paso solo reduce el intervalo a la mitad. Con los errores de `iteraciones_wien.dat` y la fórmula de la sección 4.2, el orden de Newton resultó $p \approx 2.00$ (cuadrático, duplica las cifras correctas en cada paso) y el de la secante $p \approx 1.49$, cercano al valor teórico de $1.618$. Newton necesita menos iteraciones, pero cada una evalúa la función y su derivada, mientras que la secante evalúa solo la función; por eso la secante termina costando casi lo mismo sin necesitar la derivada, y ambas son mucho más rápidas que la bisección, que a cambio es la única que siempre converge.

**2. Valor inicial.** Desde $x_0 = 0.5$, Newton llega a $x \approx 9\times10^{-17}$, es decir, a $x = 0$, en 6 iteraciones. En ese punto $F(0.5) \approx -1.47$ y $F'(0.5) \approx -2.03$, así que la tangente cruza el eje hacia la izquierda, rumbo a 0, y no hacia $x_5 \approx 4.965$. El valor $x = 0$ es una raíz correcta de la ecuación $x = 5(1 - e^{-x})$, pues ambos lados valen cero, pero no es la respuesta al problema físico: $x = hc/(\lambda k_B T) = 0$ corresponde a $\lambda \to \infty$, donde $B_\lambda \propto x^5/(e^x - 1)$ tiende a cero. Allí el espectro no tiene un máximo, y por eso el máximo de Wien es la raíz positiva $x_5$.

**3. Parada prematura.** En $[0, 100]$ sin cambio de variable, Romberg da $I \approx 1.6\times10^{-15}$ y se detiene en el nivel $k = 1$, con solo 3 evaluaciones; el valor correcto es $\pi^4/15 \approx 6.494$. En los primeros niveles solo se evalúa el integrando en $x = 0$, $x = 100$ (nivel 0) y $x = 50$ (nivel 1), y allí vale $0$, $3.7\times10^{-38}$ y $2.4\times10^{-17}$, mientras que el pico del integrando está cerca de $x \approx 2.8$ y vale $\approx 1.42$, de modo que esos puntos nunca lo "ven". Así, $S_{0,0} \approx 1.9\times10^{-36}$ y $S_{1,1} \approx 1.6\times10^{-15}$ son casi cero, su diferencia es menor que la tolerancia $10^{-8}$ y el criterio de la diagonal se cumple aunque la integral no haya convergido. Una comprobación adicional sería exigir un nivel mínimo antes de aceptar la parada (por ejemplo $k \ge 4$) o que el criterio se cumpla en dos niveles seguidos, y comparar el resultado con el obtenido con el cambio de variable.

**4. Eslabón más débil.** Al reducir $\Delta\lambda$ a la mitad, el error de (b) baja con razones $3.80$, $3.98$ y $4.00$, es decir, con orden 2; el de (c) baja mucho más rápido, con razones $37.4$, $11.1$ y $9.3$, que no llegan a un valor fijo en estas mallas. La precisión de (b) la limita la derivada central, que tiene orden 2 y es la que alimenta al spline: aunque el spline cúbico interpole mejor, el resultado no puede ser más preciso que los datos con que se ajusta. En (c) el spline se ajusta directamente al espectro, sin pasar por la derivada central, y por eso su error es mucho menor (por ejemplo, $3.1\times10^{-4}$ nm frente a $0.21$ nm con $\Delta\lambda = 12.5$ nm). El error de (a) se queda en $2.04$ nm en las cuatro mallas porque el máximo del arreglo solo puede ser un punto de la malla, y $\lambda = 500$ nm está en todas ellas y es el más cercano al máximo real ($502.04$ nm); mientras ese punto siga siendo el mayor, refinar la malla no cambia el resultado.

**5. Dos máximos.** $B_\lambda$ y $B_\nu$ no son la misma curva: una reparte la energía por unidad de longitud de onda y la otra por unidad de frecuencia, y como $\nu = c/\lambda$ el cambio de variable introduce un factor $\lambda^2$ que desplaza el pico. Por eso el espectro por longitud de onda va como $x^5/(e^x - 1)$ y el de frecuencia como $x^3/(e^x - 1)$, con raíces distintas ($x_5 \approx 4.965$ y $x_3 \approx 2.821$) y, para el Sol, picos en $502.04$ nm y $883.48$ nm. No hay un único "color" del Sol: el máximo depende de la variable con que se exprese el espectro. El de $\lambda$ ($502$ nm) está en la zona visible, mientras que el de frecuencia ($883$ nm) cae en el infrarrojo cercano; el de longitud de onda es el que se usa por convención en la ley de Wien.

### F. Verificación

La celda siguiente **no debe modificarse**. Lee los archivos que escribieron sus programas, recalcula de forma independiente los valores de referencia, y comprueba la forma de cada archivo. También avisa si un archivo parece anterior a su código. El taller exige los cinco PASS.

In [9]:
# ==================== VERIFICACIÓN DEL TALLER 8 Y 9 (no modificar) ====================
import os
import numpy as np
from scipy.interpolate import CubicSpline as _CS

H_, C_, KB_ = 6.62607015e-34, 299792458.0, 1.380649e-23

def raiz_referencia(n):
    x = float(n)
    for _ in range(200):
        x = n*(1 - np.exp(-x))
    return x

X5, X3 = raiz_referencia(5), raiz_referencia(3)
B_REF = H_*C_/(KB_*X5)
I_REF = np.pi**4/15
S_REF = 2*np.pi*KB_**4/(H_**3*C_**2)*I_REF
ITER_ESPERADAS = {(5,1):37, (5,2):4, (5,3):5,
                  (3,1):37, (3,2):5, (3,3):6}

_avisados = set()
def tabla(dat, fuentes, forma=None):
    for ruta in [dat, *fuentes]:
        if not os.path.isfile(ruta):
            raise ValueError(f"no existe {ruta}; ejecute desde la carpeta del taller")
    antiguos = [f for f in fuentes if os.path.getmtime(dat) < os.path.getmtime(f)]
    if antiguos and dat not in _avisados:
        print(f"[AVISO] {dat} parece anterior a su fuente; vuelva a generar si editó el código.")
        _avisados.add(dat)
    t = np.loadtxt(dat, ndmin=2)
    if forma is not None and t.shape != forma:
        raise ValueError(f"{dat}: forma {t.shape}; se espera {forma}")
    if not np.all(np.isfinite(t)):
        raise ValueError(f"{dat}: contiene NaN o infinitos")
    return t

def enteros(a, nombre):
    if not np.all(a == np.floor(a)):
        raise ValueError(f"{nombre}: se requieren valores enteros")

def resultados_wien():
    t = tabla('resultado_wien.dat', ['modulo_ceros.f90', 'programa_wien.f90'], (7,4))
    enteros(t[:,[0,1,3]], 'n, método e iteraciones')
    claves = [tuple(map(int, fila[:2])) for fila in t[:6]]
    if len(set(claves)) != 6 or set(claves) != set(ITER_ESPERADAS):
        raise ValueError('debe aparecer una vez cada combinación de n=5,3 y método=1,2,3')
    if tuple(t[6,:2]) != (5,4):
        raise ValueError('la fila 7 debe identificar n=5, método=4')
    if np.any((t[:,3] < 1) | (t[:,3] > 100)):
        raise ValueError('conteos fuera de 1 a 100')
    return t

def constantes():
    t = tabla('constante_wien.dat', ['modulo_ceros.f90', 'programa_wien.f90'], (1,3))[0]
    if abs(t[0]-X5)>2e-10 or abs(t[1]-X3)>2e-10:
        raise ValueError('x5 o x3 incorrectos en constante_wien.dat')
    if abs(t[2]-B_REF)/B_REF > 1e-9:
        raise ValueError('constante de Wien incorrecta')
    return t

resultados = []
def caso(nombre, funcion):
    try:
        detalle = funcion()
        ok = True
    except Exception as e:
        ok, detalle = False, str(e)
    resultados.append(ok)
    print(f"[{'PASS' if ok else 'FAIL'}] {nombre}" + (f" -> {detalle}" if detalle else ''))

def c1():
    t = resultados_wien()
    for n, m, raiz, it in t[:6]:
        ref = X5 if n == 5 else X3
        if abs(raiz-ref)>2e-10:
            raise ValueError(f'n={n:g}, método={m:g}: raíz incorrecta')
        esperado = ITER_ESPERADAS[int(n),int(m)]
        if it != esperado:
            raise ValueError(f'n={n:g}, método={m:g}: {it:g} iteraciones; se esperan {esperado}')
    return f'x5={X5:.10f}, x3={X3:.10f}; conteos correctos'

def historia_referencia(m):
    # Recurrencias independientes para verificar los registros de n=5.
    f = lambda x: x-5*(1-np.exp(-x))
    df = lambda x: 1-5*np.exp(-x)
    a,b,prev = 1.0,10.0,10.0
    x = {1:0., 2:10., 3:9.}[m]
    valores=[]
    for _ in range(ITER_ESPERADAS[5,m]):
        if m==1:
            z=(a+b)/2
            if f(a)*f(z)<=0: b=z
            else: a=z
        elif m==2: z=x-f(x)/df(x)
        else: z=x-f(x)*(x-prev)/(f(x)-f(prev))
        valores.append(z)
        prev,x=x,z
    return np.array(valores)

def c2():
    t = tabla('iteraciones_wien.dat', ['modulo_ceros.f90','programa_wien.f90'])
    if t.shape[1]!=3: raise ValueError('iteraciones_wien.dat debe tener tres columnas')
    enteros(t[:,:2], 'método y contador')
    if set(t[:,0])!={1,2,3}: raise ValueError('faltan métodos o hay identificadores inválidos')
    for m in range(1,4):
        filas=t[t[:,0]==m]
        ref=historia_referencia(m)
        if len(filas)!=len(ref): raise ValueError(f'método {m}: número de registros incorrecto')
        if not np.array_equal(filas[:,1], np.arange(1,len(ref)+1)):
            raise ValueError(f'método {m}: use contadores consecutivos desde 1, en orden')
        if not np.allclose(filas[:,2],ref,atol=2e-13,rtol=2e-14):
            raise ValueError(f'método {m}: trayectoria incorrecta o pocas cifras en el archivo')
    return 'tres historiales completos'

def c3():
    x5,x3,b=constantes()
    t=resultados_wien()
    if abs(t[6,2])>1e-8: raise ValueError('Newton desde 0.5 debe llegar a la raíz trivial')
    return f'b={b:.9e} m K; raíz trivial comprobada'

def c4():
    t=tabla('resultado_sigma.dat', ['modulo_romberg.f90','programa_sigma.f90'], (2,4))
    enteros(t[:,:2], 'nivel y evaluaciones de Romberg')
    for k,ne,I,s in t:
        if k<0 or k>50 or ne<2:
            raise ValueError('nivel o conteo de Romberg inválido')
        if int(ne)!=2**int(k)+1:
            raise ValueError('use nivel k=0 para el trapecio inicial y cuente nodos únicos: 2**k+1')
        factor=2*np.pi*KB_**4/(H_**3*C_**2)
        if not np.isclose(s,factor*I,rtol=1e-8,atol=1e-30):
            raise ValueError('sigma no corresponde a la integral de su fila')
    if abs(t[0,2]-I_REF)>1e-8 or abs(t[0,3]-S_REF)/S_REF>1e-8:
        raise ValueError('la integral transformada o sigma no alcanzan la precisión solicitada')
    prematura=100/6*(4*50**3/np.expm1(50)+100**3/np.expm1(100))
    if t[1,0]!=1 or not np.isclose(t[1,2],prematura,rtol=1e-6,atol=1e-18):
        raise ValueError('fila 2: debe mostrar la parada prematura en k = 1')
    return f'sigma={t[0,3]:.9e} W/m²/K⁴; parada prematura reproducida'

def c5():
    constantes()
    t=tabla('maximo_espectro.dat', ['maximo_espectro.py','constante_wien.dat'], (4,4))
    if not np.array_equal(t[:,0],[100.,50.,25.,12.5]):
        raise ValueError('use, en este orden, pasos de 100, 50, 25 y 12.5 nm')
    if np.any(t[:,1:]<0): raise ValueError('los errores absolutos no pueden ser negativos')
    ref=B_REF/5772*1e9
    for fila in t:
        dl=fila[0]
        l=np.arange(100.,3000.+dl/2,dl)
        B=1/l**5/np.expm1(H_*C_/(l*1e-9*KB_*5772))
        la=l[np.argmax(B)]
        def cero(s):
            r=s.roots(extrapolate=False)
            r=r[np.isfinite(r)]
            r=r[s(r,1)<0]
            if len(r)==0: raise ValueError('no se encontró candidato a máximo')
            return r[np.argmin(abs(r-la))]
        esperados=[abs(la-ref),
            abs(cero(_CS(l[1:-1],(B[2:]-B[:-2])/(2*dl)))-ref),
            abs(cero(_CS(l,B).derivative())-ref)]
        for nombre,val,esp in zip(['a','b','c'],fila[1:],esperados):
            if abs(val-esp)>1e-4+0.01*esp:
                raise ValueError(f'paso {dl:g} nm, método {nombre}: error incompatible')
    if not os.path.isfile('espectro_solar.png'):
        raise ValueError('falta espectro_solar.png')
    return 'cuatro mallas correctas'

caso('1. Raíces e iteraciones de los tres métodos', c1)
caso('2. Historiales de iteración y convergencia', c2)
caso('3. Constante de Wien y Newton desde x0=0.5', c3)
caso('4. Stefan–Boltzmann y parada prematura', c4)
caso('5. Máximo del espectro solar en cuatro mallas', c5)
print(f'\nResultado: {sum(resultados)} de {len(resultados)} PASS')

[PASS] 1. Raíces e iteraciones de los tres métodos -> x5=4.9651142317, x3=2.8214393721; conteos correctos
[PASS] 2. Historiales de iteración y convergencia -> tres historiales completos
[PASS] 3. Constante de Wien y Newton desde x0=0.5 -> b=2.897771955e-03 m K; raíz trivial comprobada
[PASS] 4. Stefan–Boltzmann y parada prematura -> sigma=5.670374419e-08 W/m²/K⁴; parada prematura reproducida
[PASS] 5. Máximo del espectro solar en cuatro mallas -> cuatro mallas correctas

Resultado: 5 de 5 PASS
